# Notebook 05 -- Recommender evaluation

**Plan 4 Task 17, Step 2.** Runs the two big evaluation harnesses against the placeholder data, produces the metric tables + 5 figures the CREST report Section 8 needs.

- `run_retrospective_backtest` -- PPI vs CROPSAP outbreaks vs seasonal + ICAR-CISH baselines.
- `evaluate_rasff_counterfactual` -- prevention-rate against historical RASFF rejections.

Outputs:
- `artifacts/eval_metrics.json` -- merged metrics dict.
- `artifacts/figs/ppi_vs_outbreak_roc.png`
- `artifacts/figs/rasff_prevention_rate.png`
- `artifacts/figs/baseline_comparison.png`
- `artifacts/figs/per_pathogen_confusion.png`
- `artifacts/figs/per_segment_breakdown.png`

These are CREST report Section 8 fixtures. Replace the synthetic retrospective table with the real curated dataset before submission.

In [ ]:
from __future__ import annotations

import json
from dataclasses import fields
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve

from mangoguard.evaluation.rasff_counterfactual import (
    evaluate_rasff_counterfactual,
    evaluate_with_kvk_baseline,
)
from mangoguard.evaluation.retrospective import run_retrospective_backtest
from mangoguard.recommend import rasff as rasff_module

REPO_ROOT = Path("..").resolve()
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
FIGS_DIR = ARTIFACTS_DIR / "figs"
ARTIFACTS_DIR.mkdir(exist_ok=True)
FIGS_DIR.mkdir(exist_ok=True)

## 1. Retrospective backtest (PPI vs CROPSAP outbreaks)

PLACEHOLDER: replace synthetic data with the real 2018-2024 (block, day, PPI) + (block, day, outbreak) tables.

In [ ]:
rng = np.random.default_rng(11)
n_blocks = 5
n_days = 120
obs_rows = []
for b in range(n_blocks):
    for d in range(n_days):
        ppi = float(rng.uniform(0, 100))
        obs_rows.append(
            {
                "block_id": f"block{b}",
                "date": pd.Timestamp("2024-03-01") + pd.Timedelta(days=d),
                "ppi": ppi,
            }
        )
obs = pd.DataFrame(obs_rows)
p = (obs["ppi"] / 100.0).clip(0, 1)
outbreaks = (rng.uniform(size=len(obs)) < p * 0.55).astype(int) | (obs["ppi"] > 70).astype(int)
out = pd.DataFrame(
    {
        "block_id": obs["block_id"],
        "date": obs["date"],
        "outbreak": outbreaks.astype(int),
    }
)
result = run_retrospective_backtest(obs, out)
result

In [ ]:
# Figure 1: ROC curve
merged = obs.merge(out, on=["block_id", "date"])
y_true = merged["outbreak"]
y_score = (merged["ppi"] / 100.0).clip(0, 1)
fpr, tpr, _ = roc_curve(y_true, y_score)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f'MangoGuard PPI (AUC={result["mangoguard"]["roc_auc"]:.3f})')
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("PPI vs CROPSAP outbreak ROC")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGS_DIR / "ppi_vs_outbreak_roc.png", dpi=150)
plt.show()

In [ ]:
# Figure 2: Per-baseline AUC comparison
labels = ["MangoGuard", "Seasonal mean", "ICAR-CISH calendar"]
aucs = [
    result["mangoguard"]["roc_auc"],
    result["seasonal_baseline"]["roc_auc"],
    result["icar_cish_baseline"]["roc_auc"],
]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, aucs, color=["#2a9d8f", "#e9c46a", "#e76f51"])
for bar, auc in zip(bars, aucs, strict=False):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{auc:.3f}",
        ha="center",
        fontsize=10,
    )
ax.set_ylabel("ROC-AUC")
ax.set_title("Disease-pressure discrimination -- MangoGuard vs baselines")
ax.set_ylim(0, 1)
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(FIGS_DIR / "baseline_comparison.png", dpi=150)
plt.show()

## 2. RASFF counterfactual

Replays each historical RASFF rejection through MangoGuard's recommender + each calendar baseline.

In [ ]:
rasff_rows = rasff_module.all_rows()
field_names = [f.name for f in fields(rasff_rows[0])]
rasff_df = pd.DataFrame([{n: getattr(row, n) for n in field_names} for row in rasff_rows])
print(f"Loaded {len(rasff_df)} RASFF rejections")

icar = evaluate_rasff_counterfactual(rasff_df, cutoff=0.10)
kvk = evaluate_with_kvk_baseline(rasff_df, cutoff=0.10)
print("ICAR-CISH baseline:", icar)
print("KVK Konkan baseline:", kvk)

In [ ]:
# Figure 3: RASFF prevention-rate comparison
labels = ["MangoGuard", "ICAR-CISH", "KVK Konkan"]
rates = [
    icar["prevention_rate_mangoguard"],
    icar["prevention_rate_icar_baseline"],
    kvk["prevention_rate_kvk_baseline"],
]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, rates, color=["#2a9d8f", "#e9c46a", "#f4a261"])
for bar, r in zip(bars, rates, strict=False):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{r:.1%}",
        ha="center",
        fontsize=10,
    )
ax.set_ylabel("Prevention rate")
ax.set_title("RASFF counterfactual -- fraction of rejections prevented")
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(FIGS_DIR / "rasff_prevention_rate.png", dpi=150)
plt.show()

In [ ]:
# Figure 4: Per-pathogen confusion
ppi_pred = (merged["ppi"] >= 50).astype(int)
merged_with_pred = merged.assign(predicted=ppi_pred)
conf = pd.crosstab(
    merged_with_pred["outbreak"],
    merged_with_pred["predicted"],
    rownames=["Outbreak"],
    colnames=["Predicted spray"],
)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(conf.values, cmap="Blues")
for i in range(conf.shape[0]):
    for j in range(conf.shape[1]):
        ax.text(j, i, str(conf.values[i, j]), ha="center", va="center")
ax.set_xticks([0, 1])
ax.set_xticklabels(["No", "Yes"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["No", "Yes"])
ax.set_xlabel("Predicted spray")
ax.set_ylabel("Outbreak")
ax.set_title("Anthracnose confusion at PPI threshold 50")
fig.tight_layout()
fig.savefig(FIGS_DIR / "per_pathogen_confusion.png", dpi=150)
plt.show()

In [ ]:
# Figure 5: per-market-segment RASFF rejection probability of top historical offenders
offenders = ["chlorpyrifos", "carbendazim", "imidacloprid", "acephate"]
destinations = ["EU", "JAPAN", "US"]
matrix = []
for ai in offenders:
    row = [rasff_module.rejection_probability(ai, d) for d in destinations]
    matrix.append(row)
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(np.array(matrix), cmap="Reds", vmin=0, vmax=0.3)
for i in range(len(offenders)):
    for j in range(len(destinations)):
        ax.text(j, i, f"{matrix[i][j]:.3f}", ha="center", va="center")
ax.set_xticks(range(len(destinations)))
ax.set_xticklabels(destinations)
ax.set_yticks(range(len(offenders)))
ax.set_yticklabels(offenders)
ax.set_title("Smoothed RASFF rejection probability")
fig.colorbar(im, ax=ax, label="p_hat")
fig.tight_layout()
fig.savefig(FIGS_DIR / "per_segment_breakdown.png", dpi=150)
plt.show()

In [ ]:
# Persist metrics
metrics = {
    "retrospective_backtest": result,
    "rasff_counterfactual_icar": icar,
    "rasff_counterfactual_kvk": kvk,
}
with open(ARTIFACTS_DIR / "eval_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"Wrote {ARTIFACTS_DIR / 'eval_metrics.json'}")